# AML Copilot Agent Evaluation - Interactive Demo

This notebook provides an interactive interface for running and visualizing agent evaluations using the golden test framework.

**Purpose:**
- Demonstrate agent capabilities to stakeholders
- Explore agent behavior on specific test cases
- Visualize evaluation metrics
- Debug and analyze agent outputs

**Note:** For automated CI/CD testing, use `tests/evaluation/test_runner.py` directly.

In [ ]:
from pathlib import Path
import json
from IPython.display import display, HTML, Markdown
import pandas as pd

from tests.evaluation.test_runner import AgentEvaluationRunner
from tests.evaluation.models import GoldenTestCase
from tests.config import GOLDEN_DATASETS_DIR

## 1. Initialize Evaluation Runner

In [ ]:
# Initialize runner with default configuration
runner = AgentEvaluationRunner()

print("✅ Evaluation runner initialized")
print(f"   Graph compiled with {len(runner.graph.nodes)} nodes")

## 2. Load Golden Test Cases

Available datasets:
- `structuring_cases.json` - Structuring/threshold avoidance scenarios
- More datasets coming soon...

In [ ]:
# Load structuring test cases
dataset_path = GOLDEN_DATASETS_DIR / "structuring_cases.json"
test_cases = runner.load_golden_test_cases(dataset_path)

print(f"Loaded {len(test_cases)} test cases:")
for tc in test_cases:
    print(f"  • {tc.test_id}: {tc.description} [Priority: {tc.priority}]")

## 3. Run Single Test Case (Detailed View)

Execute a single test case to see detailed agent behavior.

In [ ]:
# Select test case to run
test_case = test_cases[0]  # STRUCT_001

print(f"Test Case: {test_case.test_id}")
print(f"Description: {test_case.description}")
print(f"\nUser Query: \"{test_case.input.user_query}\"")
print(f"\nContext:")
print(json.dumps(test_case.input.context, indent=2))

In [ ]:
# Execute test case
result = runner.execute_test_case(test_case)

### Agent Output

In [ ]:
# Display agent's compliance analysis
compliance_analysis = result.agent_output.get("compliance_analysis", {})

print("=" * 70)
print("AGENT COMPLIANCE ANALYSIS")
print("=" * 70)

if compliance_analysis:
    print(f"\n📊 Typologies Identified: {compliance_analysis.get('typologies', [])}")
    print(f"⚠️  Risk Assessment: {compliance_analysis.get('risk_assessment', 'N/A')}")
    print(f"\n📋 Analysis:\n")
    print(compliance_analysis.get('analysis', 'No analysis available'))
    
    if compliance_analysis.get('recommendations'):
        print(f"\n✅ Recommendations:")
        for i, rec in enumerate(compliance_analysis['recommendations'], 1):
            print(f"   {i}. {rec}")
    
    if compliance_analysis.get('regulatory_references'):
        print(f"\n📖 Regulatory References:")
        for ref in compliance_analysis['regulatory_references']:
            print(f"   • {ref}")
else:
    print("No compliance analysis available")

### Evaluation Results

In [ ]:
# Create evaluation summary
status_emoji = "✅" if result.status == "PASS" else "❌" if result.status == "FAIL" else "⚠️"

print(f"\n{status_emoji} Test Status: {result.status}")
print(f"\n📈 Scores:")
print(f"   Overall: {result.overall_score:.1f}/100")
print(f"   Correctness: {result.correctness_score:.2f}")
print(f"   Completeness: {result.completeness_score:.2f}")
print(f"   Hallucination: {result.hallucination_score:.2f}")
print(f"\n⏱️  Execution Time: {result.execution_time_seconds:.2f}s")

if result.pass_reasons:
    print(f"\n✓ Pass Reasons:")
    for reason in result.pass_reasons:
        print(f"  • {reason}")

if result.fail_reasons:
    print(f"\n✗ Fail Reasons:")
    for reason in result.fail_reasons:
        print(f"  • {reason}")

if result.key_facts_covered:
    print(f"\n✓ Key Facts Covered ({len(result.key_facts_covered)}):")
    for fact in result.key_facts_covered:
        print(f"  • {fact}")

if result.key_facts_missing:
    print(f"\n✗ Key Facts Missing ({len(result.key_facts_missing)}):")
    for fact in result.key_facts_missing:
        print(f"  • {fact}")

### Ground Truth Comparison

In [ ]:
# Compare to expected output
expected = test_case.expected_output

comparison_data = {
    "Aspect": [
        "Typologies",
        "Red Flags",
        "Risk Assessment",
        "Key Facts Coverage"
    ],
    "Expected": [
        ", ".join(expected.typologies_identified) if expected else "N/A",
        ", ".join(expected.red_flags_identified) if expected else "N/A",
        expected.risk_assessment if expected else "N/A",
        f"{len(expected.key_facts_mentioned)} facts" if expected else "N/A"
    ],
    "Agent Output": [
        ", ".join(compliance_analysis.get('typologies', [])),
        "See analysis",
        compliance_analysis.get('risk_assessment', 'N/A'),
        f"{len(result.key_facts_covered)}/{len(expected.key_facts_mentioned)} covered" if expected else "N/A"
    ],
    "Match": [
        "✅" if any(t in compliance_analysis.get('typologies', []) for t in expected.typologies_identified) else "❌" if expected else "N/A",
        "✅" if len(result.key_facts_covered) > 0 else "❌",
        "✅" if compliance_analysis.get('risk_assessment') == expected.risk_assessment else "❌" if expected else "N/A",
        "✅" if result.completeness_score >= 0.8 else "❌"
    ]
}

df_comparison = pd.DataFrame(comparison_data)
display(df_comparison)

## 4. Run Full Evaluation Suite

Execute all test cases and generate aggregate metrics.

In [ ]:
# Run full suite
report = runner.run_evaluation_suite(dataset_path)

### Aggregate Metrics Visualization

In [ ]:
# Create summary DataFrame
results_data = [
    {
        "Test ID": r.test_id,
        "Status": r.status,
        "Overall Score": f"{r.overall_score:.1f}",
        "Correctness": f"{r.correctness_score:.2f}",
        "Completeness": f"{r.completeness_score:.2f}",
        "Hallucination": f"{r.hallucination_score:.2f}",
        "Time (s)": f"{r.execution_time_seconds:.2f}"
    }
    for r in report.test_results
]

df_results = pd.DataFrame(results_data)
display(df_results)

In [ ]:
# Summary statistics
print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)
print(f"\nPass Rate: {report.pass_rate:.1%} ({report.passed}/{report.total_cases})")
print(f"\nAverage Scores:")
print(f"  Overall: {report.avg_overall_score:.1f}/100")
print(f"  Correctness: {report.avg_correctness_score:.2f}")
print(f"  Completeness: {report.avg_completeness_score:.2f}")
print(f"  Hallucination: {report.avg_hallucination_score:.2f}")

if report.action_items:
    print(f"\n📋 Action Items:")
    for item in report.action_items:
        print(f"  • {item}")

## 5. Save Report

Save evaluation report for future reference or regression comparison.

In [ ]:
# Save report
from tests.config import EVALUATION_REPORTS_DIR

output_path = EVALUATION_REPORTS_DIR / f"{report.report_id}.json"
runner.save_report(report, output_path)

print(f"✅ Report saved to: {output_path}")

## 6. Custom Exploration

Use this section to explore specific test cases or scenarios.

In [ ]:
# Example: Filter by priority
high_priority_cases = runner.load_golden_test_cases(dataset_path, priority="HIGH")
print(f"High priority test cases: {len(high_priority_cases)}")

for tc in high_priority_cases:
    print(f"  • {tc.test_id}: {tc.description}")

In [ ]:
# Example: Inspect ML model output for a test case
test_case = test_cases[0]
ml_output = test_case.input.ml_output

print("ML Model Output:")
print(json.dumps(ml_output, indent=2))

## Notes

**For Development:**
- Use this notebook to explore and debug agent behavior
- Visualize test results for stakeholder demos
- Analyze specific test case failures

**For CI/CD:**
- Use `tests/evaluation/test_runner.py` directly in automated pipelines
- Save reports to `tests/evaluation/reports/` for regression tracking
- Set up quality gates based on pass rate and metrics

**Next Steps:**
1. Add visualization libraries (matplotlib, plotly) for metric charts
2. Create comparative analysis between versions
3. Add interactive widgets for test case selection
4. Export results to presentation-ready formats (PDF, HTML)